<span id="영상-검출에서-안전한-제어-명령으로"></span>

# From Image Detection to Safe Control Commands

<span id="학습-목표"></span>

## Learning objectives

Use simulated inputs to examine how sign errors and oscillation arise when horizontal and vertical image errors are converted into movement commands.

<span id="실행-조건"></span>

## Execution conditions

The basic activities use NumPy and Matplotlib. Use OpenCV when connecting to the previous unit.

<span id="검출-위치를-오차로-바꾸기"></span>

## Convert detected positions to errors

<span id="시작-조건"></span>

### Starting conditions

The center of a 160×120 image is (80,60), and the observed center is (110,50). x points right and y points down.

<span id="진행-방법"></span>

### Procedure

1.  Calculate $e_x=x-80$ and $e_y=y-60$ using the same sign convention as `command(center)` in the example. Plot the image center and observed point, and label both errors on an arrow from the center to the observed point.
2.  For each axis, set the command to 0 if $|e|\le10$ pixels; otherwise calculate it using $0.02e$, then limit it to $[-0.5,0.5]$. Calculate both axis commands for $(110,50)$ and record them beside the function outputs.

<span id="남길-결과"></span>

### Results to retain

- A center/error arrow diagram and example commands.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def command(center):
    if center is None:return np.zeros(2)
    error=np.asarray(center)-[80,60]
    return np.clip(np.where(np.abs(error)<=10,0,.02*error),-.5,.5)
print(command((110,50)),command(None))
plt.scatter([80,110],[60,50]);plt.arrow(80,60,30,-10,head_width=3);plt.xlim(0,160);plt.ylim(120,0);plt.gca().set_aspect('equal')


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**Observed center:**

**x/y errors:**

**Validity:**

**Command:**

<span id="확인할-점"></span>

### Points to check

- Have you distinguished the image y-axis direction from upward in world coordinates?
- Did you distinguish pixel error and normalized commands from the aircraft's actual velocity \[m/s\]?

<span id="흔들림과-미검출-처리"></span>

## Handle fluctuations and missed detections

<span id="시작-조건-1"></span>

### Starting conditions

x observations are \[80,82,79,110,None,108\], with $y=60$. None denotes a frame where the object was not found.

<span id="진행-방법-1"></span>

### Procedure

1.  For each valid observation, calculate the command by applying `command((x,60))` directly, and separately by applying it to the mean x of up to the 3 most recent valid observations. At startup, if fewer than 3 samples are available, average only the collected samples.
2.  At `None`, set the command to 0 and clear the averaging history. Treat the later 108 as the first sample in a new history.
3.  Record frame order, raw observations, mean x, and both commands in a table and plot them against the same frame axis. Explain the differences by identifying both commands at the frame where x changes to 110 and the command immediately after a missed detection.

<span id="남길-결과-1"></span>

### Results to retain

- Observation/command time series and missed detection rules.

In [ ]:
from collections import deque
history=deque(maxlen=3)
for x in [80,82,79,110,None,108]:
    if x is None:history.clear();filtered=None
    else:history.append(x);filtered=(np.mean(history),60)
    print(x,filtered,command(filtered))
# TODO: Plot raw and filtered commands on the same axes.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**Frame:**

**Observed x:**

**Filtered x:**

**Command:**

**Handling missed detections:**

<span id="확인할-점-1"></span>

### Points to check

- Did you also check that a moving average causes response delay?
- Have you avoided including None as numeric 0 in the average?

<span id="검출부터-명령까지-연결하기"></span>

## Connect detection to commands

<span id="시작-조건-2"></span>

### Starting conditions

Connect the previous unit's mask center calculation to command. The basic activities use only saved or generated images.

<span id="진행-방법-2"></span>

### Procedure

1.  Separate the function returning center, area, and detection time from the command function.
2.  Test an empty mask, a valid region, and a frame more than 0.2 seconds past detection, in that order. Identify stale frames by the difference between the current time and frame detection time, and set commands to 0 in that case.
3.  Describe additional coordinate transformations, velocity limits, and stop conditions needed for a real device.

<span id="남길-결과-2"></span>

### Results to retain

- An input→detection→validation→command flowchart and an error case table.

In [ ]:
# TODO: Connect detect_center(frame) and command(center), and
# Set the command to 0 if the Frame timestamp is more than .2 s old.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Input frame:**

**Detection validity:**

**Command:**

**Stop reason:**

<span id="확인할-점-2"></span>

### Points to check

- Does a failed detection clear the previous velocity command?
- Can you test error paths without actual flight?

<span id="공식-예제데이터"></span>

## Official examples and data

- [OpenCV · Color spaces and region extraction](https://docs.opencv.org/4.x/df/d9d/tutorial_py_colorspaces.html)
- [OpenCV · Circle detection](https://docs.opencv.org/4.x/da/d53/tutorial_py_houghcircles.html)